# EXP_030B — Text Ablation: Best Image + PhoBERT + Concat + MSE
**Phase 3 | Text Backbone Ablation**
Research question: Does PhoBERT (Vietnamese-specific) improve over XLM-R for Foody reviews?
- Text model: `vinai/phobert-base-v2` | Image model: Best from Phase 2 (set below)
- Fusion: Concatenation + MLP | Loss: MSE | Seed: 42 | AMP: enabled
> ⚠️ Prerequisite: Phase 2 (EXP_020B or EXP_020D) must be completed. Set BEST_IMAGE_MODEL in STEP 4.

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!gdown --id 11WoeUn2visKtGN5oOX9c2I6Grz3P88vD -O data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

### STEP 4: Configure paths — ✏️ Set Phase 2 winner here

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_030B_bestimage_phobert_concat_mse'

# ✏️ SET based on Phase 2 results (lowest Mean MAE wins)
# Options: 'swin_base_patch4_window7_224' / 'efficientnet_b3' / 'convnext_base_in22k'
BEST_IMAGE_MODEL  = 'swin_base_patch4_window7_224'
# Options: 'EXP_020B_swinb_xlmr_concat_mse' / 'EXP_020D_efficientnetb3_xlmr_concat_mse' / 'EXP_012_multimodal_convnext_xlmr_concat_mse'
BEST_IMAGE_EXP_ID = 'EXP_020B_swinb_xlmr_concat_mse'

DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts will be saved to: {DRIVE_EXP_PATH}')
print(f'Using image backbone  : {BEST_IMAGE_MODEL}')
print(f'Loading image weights : {BEST_IMAGE_EXP_ID}')

### STEP 5: Load image weights from Phase 2 winner (PhoBERT trains from HuggingFace pretrained weights)

In [ ]:
import os, shutil
os.makedirs('./checkpoints', exist_ok=True)
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_IMAGE_EXP_ID}/best_model.pth', './checkpoints/best_model_train_image.pth')
print(f'Loaded image weights from {BEST_IMAGE_EXP_ID}')
if os.path.exists('./checkpoints/best_model_train_text.pth'):
    os.remove('./checkpoints/best_model_train_text.pth')
    print('Removed old text checkpoint — PhoBERT uses HuggingFace pretrained weights')

### STEP 6: Train

In [ ]:
!python main.py \
  --mode train_fusion \
  --text_model_name vinai/phobert-base-v2 \
  --image_model_name {BEST_IMAGE_MODEL} \
  --epochs 15 \
  --batch_size 8 \
  --lr 1e-5 \
  --grad_accum_steps 4 \
  --patience 5 \
  --loss_fn mse \
  --unfreeze_text_layers 1 \
  --unfreeze_image_layers 1 \
  --seed 42 \
  --use_amp \
  --exp_id EXP_030B_bestimage_phobert_concat_mse \
  --exp_dir ./experiments

### STEP 7: Save to Drive + print metrics

In [ ]:
import shutil, json
src = f'./experiments/{EXP_ID}'
shutil.copytree(src, DRIVE_EXP_PATH, dirs_exist_ok=True)
print(f'Saved to {DRIVE_EXP_PATH}')
with open(f'{src}/metrics.json') as f:
    m = json.load(f)
print(f"\n=== EXP_030B Results ===")
print(f"Mean MAE: {m['mean_mae']:.4f} | Overall MAE: {m['overall_mae']:.4f}")
for k in ['food','price','atmos','service']: print(f"  {k}: {m[f'mae_{k}']:.4f}")